In [ ]:
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv

load_dotenv(Path('..') / '.env')
sys.path.insert(0, str(Path('.')))

from growth_analysis.acquisition import acquisition_metrics, channel_comparison

In [ ]:
# Constants
DATA_DIR = Path('..') / 'data' / 'synthetic'
LOOKBACK_DAYS = 90

In [ ]:
users  = pd.read_csv(DATA_DIR / 'users.csv',  parse_dates=['signup_at', 'churn_at'])
orders = pd.read_csv(DATA_DIR / 'orders.csv', parse_dates=['billed_at', 'period_start', 'period_end'])
print(f"{len(users):,} users  |  {len(orders):,} orders")

In [ ]:
# Channel-level acquisition metrics
metrics = acquisition_metrics(users, orders)
metrics

In [ ]:
# Visualise conversion rate and ARPU per channel
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = plt.cm.Set2.colors

# Conversion rate
axes[0].bar(metrics['acquisition_channel'], metrics['conversion_rate'] * 100, color=colors)
axes[0].set_ylabel('Conversion rate (%)')
axes[0].set_title('Conversion Rate by Channel')
axes[0].tick_params(axis='x', rotation=30)

# ARPU (if revenue data present)
if 'arpu' in metrics.columns:
    axes[1].bar(metrics['acquisition_channel'], metrics['arpu'].fillna(0), color=colors)
    axes[1].set_ylabel('ARPU (USD)')
    axes[1].set_title('ARPU by Channel')
    axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

In [ ]:
# Monthly signups trend by channel
trend = channel_comparison(users)

fig, ax = plt.subplots(figsize=(12, 5))
for channel, grp in trend.groupby('acquisition_channel'):
    ax.plot(grp['period'], grp['signups'], marker='o', label=channel)

ax.set_xlabel('Month')
ax.set_ylabel('Signups')
ax.set_title('Monthly Signups by Acquisition Channel')
ax.tick_params(axis='x', rotation=30)
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()